# 02 — Sets, Hash Tables, and Set Algebra

## Theoretical Foundations of Sets

In discrete mathematics, a **set** is an unordered collection of distinct, well-defined elements. Formally, a set $S$ is characterized by its characteristic membership function $\chi_S$:

$$\chi_S(x) = \begin{cases} 1 & \text{if } x \in S \\ 0 & \text{if } x \notin S \end{cases}$$

In Python, the `set` type is implemented as a high-performance **hash table** (`PySetObject`). Each entry stores an element's hash value and an object pointer. Because index position is governed entirely by the element's hash code modulo table capacity:
- Lookups, insertions, and deletions execute in **average $O(1)$ constant time**.
- Duplicate entries are strictly rejected.
- Sets are unordered and do not support positional indexing (`s[0]` is invalid).

## The Hashability Contract

To be stored within a set, an object must be **hashable**:
1. It must provide a `__hash__()` method returning an integer invariant throughout the object's lifetime.
2. It must provide an `__eq__()` method to resolve hash collisions.
3. If two objects compare equal (`a == b`), they must produce identical hash values (`hash(a) == hash(b)`).

Immutable built-in types (`int`, `float`, `str`, `tuple`, `frozenset`) are hashable. Mutable containers (`list`, `dict`, `set`) are unhashable and raise `TypeError`.


In [ ]:
# 1. Hashability Contract Demonstration
# Hashable primitives
print(f"hash(42):          {hash(42)}")
print(f"hash('token'):     {hash('token')}")
print(f"hash((10, 20)):    {hash((10, 20))}")

# Creating valid sets
valid_set = {1, 2.5, "feature", (0, 1)}
print(f"Valid set: {valid_set}, size: {len(valid_set)}")

# Attempting to store an unhashable mutable object (list) in a set
try:
    invalid_set = {[1, 2], [3, 4]}  # type: ignore[var-annotated]
except TypeError as error:
    print(f"Caught expected unhashable error: {error}")

# Empty set instantiation: {} creates a dict, set() creates a set
empty_dict = {}
empty_set = set()
print(f"Type of {{}}:     {type(empty_dict).__name__}")
print(f"Type of set(): {type(empty_set).__name__}")

assert isinstance(empty_set, set)
assert isinstance(empty_dict, dict)

## Set Mutations: Addition, Deletion, and Idempotency

Sets support dynamic element manipulation:
- `add(elem)`: Inserts an element. If already present, the operation is an idempotent no-op.
- `remove(elem)`: Deletes an element; raises `KeyError` if absent.
- `discard(elem)`: Safely deletes an element if present; silent no-op if absent.
- `pop()`: Removes and returns an arbitrary element; raises `KeyError` if empty.
- `clear()`: Empties the entire set.


In [ ]:
# 2. Set Mutation Operations
animals = {"cat", "dog", "bird"}

# Adding items
animals.add("horse")
animals.add("cat")  # Idempotent addition: no duplicates created
print(f"After additions: {animals}, length: {len(animals)}")
assert len(animals) == 4

# Safe removal with discard vs defensive remove
animals.discard("tiger")  # Does not raise error
print("discard('tiger') executed safely.")

try:
    animals.remove("tiger")
except KeyError as error:
    print(f"remove('tiger') raised expected KeyError: {error}")

# pop an arbitrary element
popped_animal = animals.pop()
print(f"Popped element: {popped_animal}")
assert popped_animal not in animals

## Mathematical Set Algebra: Operators vs. Methods

Python sets natively support algebraic set operations with dual interfaces:
1. **Operator syntax:** Requires both operands to be sets (`A | B`).
2. **Method syntax:** Accepts any iterable as the argument (`A.union([1, 2])`).

| Operation | Mathematical Notation | Operator | Method |
| :--- | :---: | :---: | :--- |
| **Union** | $A \cup B$ | `A \| B` | `A.union(B)` |
| **Intersection** | $A \cap B$ | `A & B` | `A.intersection(B)` |
| **Difference** | $A \setminus B$ | `A - B` | `A.difference(B)` |
| **Symmetric Difference** | $A \triangle B$ | `A ^ B` | `A.symmetric_difference(B)` |
| **Subset** | $A \subseteq B$ | `A <= B` | `A.issubset(B)` |
| **Superset** | $A \supseteq B$ | `A >= B` | `A.issuperset(B)` |
| **Disjoint** | $A \cap B = \emptyset$ | — | `A.isdisjoint(B)` |


In [ ]:
# 3. Set Algebra Demonstration
set_a = {1, 2, 3, 4, 5}
set_b = {4, 5, 6, 7, 8}

# Union: A u B
union_ab = set_a | set_b
assert union_ab == {1, 2, 3, 4, 5, 6, 7, 8}

# Intersection: A n B
inter_ab = set_a & set_b
assert inter_ab == {4, 5}

# Difference: A \ B (elements in A but not in B)
diff_ab = set_a - set_b
assert diff_ab == {1, 2, 3}

# Symmetric Difference: A Delta B (elements in either A or B, but not both)
sym_diff_ab = set_a ^ set_b
assert sym_diff_ab == {1, 2, 3, 6, 7, 8}

# Invariant: A Delta B == (A u B) \ (A n B)
assert sym_diff_ab == (union_ab - inter_ab)

# Relational tests
sub_set = {1, 2}
assert sub_set <= set_a
assert set_a >= sub_set
assert set_a.isdisjoint({9, 10})

print("Set algebra operations successfully validated.")

## Set Comprehensions

Similar to list comprehensions, **set comprehensions** provide concise mathematical set-builder syntax:

$$\{ f(x) \mid x \in S, P(x) \}$$

Set comprehensions automatically enforce uniqueness across generated values.


In [ ]:
# 4. Set Comprehension
corpus_tokens = ["convolution", "pooling", "stride", "padding", "convolution", "RELU", "relu"]

# Extract normalized lowercase tokens with length >= 6
normalized_vocab = {token.lower() for token in corpus_tokens if len(token) >= 6}

print(f"Original token count: {len(corpus_tokens)}")
print(f"Unique vocab >= 6 chars: {normalized_vocab}")

assert normalized_vocab == {"convolution", "pooling", "padding"}

## Case Study: Text Vocabulary Extraction and Index Mapping

In Natural Language Processing (NLP), a foundational task is reading a text corpus, stripping punctuation and typographical variations, and constructing an indexed vocabulary dictionary ($V = \{w_1, w_2, \dots, w_k\}$).

Here, we ingest `../data/data.txt`, clean the tokens, extract unique vocabulary via set algebra, and build token-to-index and index-to-token mappings.


In [ ]:
# 5. NLP Vocabulary Extraction Case Study
from pathlib import Path

# Load corpus text
text_file = Path("../data/data.txt")
raw_corpus = text_file.read_text(encoding="utf-8")
print(f"Corpus snippet: {raw_corpus[:120]}...\n")

# Text cleaning and tokenization
punctuation_to_strip = ".,;:()-/\"'"
cleaned_text = raw_corpus.lower()
for char in punctuation_to_strip:
    cleaned_text = cleaned_text.replace(char, " ")

tokens = [tok.strip() for tok in cleaned_text.split() if tok.strip()]
print(f"Total tokens in document: {len(tokens)}")

# O(n) vocabulary extraction via set construction
vocabulary_set = set(tokens)
print(f"Unique vocabulary size:   {len(vocabulary_set)}")

# Building deterministic token-to-id mapping
sorted_vocab = sorted(vocabulary_set)
token2id = {token: idx for idx, token in enumerate(sorted_vocab)}
id2token = {idx: token for token, idx in token2id.items()}

print(f"First 5 vocabulary entries: {sorted_vocab[:5]}")
assert "python" in token2id
assert id2token[token2id["python"]] == "python"
print("Vocabulary mapping constructed successfully.")